# MedForget-bench — 00: PathMNIST Dataset Exploration

**Phase 0 deliverable** — Explore PathMNIST before any ML implementation.

### Goals
1. Load PathMNIST via the `medmnist` package
2. Inspect dataset size, class balance, and image statistics
3. Visualise sample images per class
4. Confirm the dataset is suitable for class-wise forgetting
5. Check that our planned forget class (class index 3) is well-represented

### Why PathMNIST?
PathMNIST is a 9-class colon pathology dataset from [MedMNIST v2](https://medmnist.com/).
Images are 28×28 px RGB — small enough to train quickly on a laptop or free-tier Colab,
large enough to produce meaningful benchmarking results.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from torch.utils.data import DataLoader
import medmnist
from medmnist import PathMNIST, INFO

print(f'medmnist version : {medmnist.__version__}')
print(f'torch version    : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
else:
    print('No GPU locally — use Colab/Kaggle for training runs.')

## 1. Load PathMNIST

The `medmnist` package auto-downloads the dataset on first run (~100 MB total for all splits).

In [ ]:
# ── Load PathMNIST ────────────────────────────────────────────────────────────
# size=28 keeps it at the native 28×28 resolution
train_dataset = PathMNIST(split='train', download=True, size=28)
val_dataset   = PathMNIST(split='val',   download=True, size=28)
test_dataset  = PathMNIST(split='test',  download=True, size=28)

print(f'Train size : {len(train_dataset):,}')
print(f'Val size   : {len(val_dataset):,}')
print(f'Test size  : {len(test_dataset):,}')

## 2. Dataset Metadata

In [ ]:
# ── Dataset metadata from medmnist.INFO ─────────────────────────────────────
info = INFO['pathmnist']
print('Dataset name  :', info['python_class'])
print('Task type     :', info['task'])
print('Num classes   :', info['n_classes'])
print('n_channels    :', info['n_channels'])
print('Label names   :')
for idx, label in info['label'].items():
    print(f'  [{idx}] {label}')

## 3. Class Balance

For class-wise forgetting to be meaningful, each class needs enough samples.
We check our planned forget target (class 3) is well-represented.

In [ ]:
# ── Class balance in training set ─────────────────────────────────────────────
train_labels = np.array([train_dataset[i][1].item() for i in range(len(train_dataset))])
classes = list(info['label'].values())
num_classes = len(classes)

FORGET_CLASS = 3

print(f'{"Class":>5}  {"Label":<25}  {"Count":>6}  {"Pct":>5}')
print('-' * 50)
for idx in range(num_classes):
    count = (train_labels == idx).sum()
    pct = count / len(train_labels) * 100
    marker = '  <-- FORGET TARGET' if idx == FORGET_CLASS else ''
    print(f'{idx:>5}  {classes[idx]:<25}  {count:>6}  {pct:>4.1f}%{marker}')

In [ ]:
# ── Class balance bar chart ────────────────────────────────────────────────────
counts = [(train_labels == cls).sum() for cls in range(num_classes)]
bar_colors = ['#e74c3c' if idx == FORGET_CLASS else '#3498db' for idx in range(num_classes)]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(num_classes), counts, color=bar_colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(num_classes))
ax.set_xticklabels([f'{idx}\n{classes[idx][:12]}' for idx in range(num_classes)], fontsize=8)
ax.set_ylabel('Number of training samples')
ax.set_title('PathMNIST — Class Balance  (red = planned forget class)', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('pathmnist_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pathmnist_class_balance.png')

## 4. Sample Images Per Class

5 random samples per class — visualise what the model is classifying.

In [ ]:
# ── Sample images per class ────────────────────────────────────────────────────
SAMPLES_PER_CLASS = 5
np.random.seed(42)

class_indices = {cls: np.where(train_labels == cls)[0] for cls in range(num_classes)}

fig = plt.figure(figsize=(SAMPLES_PER_CLASS * 2.2, num_classes * 2.0))
fig.suptitle(
    'PathMNIST — 5 Random Samples Per Class  (red border = planned forget class)',
    fontsize=12, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(num_classes, SAMPLES_PER_CLASS, figure=fig, hspace=0.5, wspace=0.1)

for cls in range(num_classes):
    sample_idxs = np.random.choice(class_indices[cls], SAMPLES_PER_CLASS, replace=False)
    for col, idx in enumerate(sample_idxs):
        img, label = train_dataset[int(idx)]
        img = np.array(img)  # uint8 HWC from medmnist
        # medmnist returns (H, W, C) for as_rgb / default
        if img.ndim == 3 and img.shape[0] in (1, 3) and img.shape[0] < img.shape[-1]:
            img = np.transpose(img, (1, 2, 0))  # CHW -> HWC if needed

        ax = fig.add_subplot(gs[cls, col])
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_title(f'[{cls}] {classes[cls][:14]}', fontsize=6.5, loc='left', pad=2)
        if cls == FORGET_CLASS:
            for spine in ax.spines.values():
                spine.set_edgecolor('#e74c3c')
                spine.set_linewidth(2.5)
                spine.set_visible(True)

plt.savefig('pathmnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pathmnist_samples.png')

## 5. Image Statistics

Per-channel mean and std across the training set.
Used in `transforms.Normalize(mean=..., std=...)` when building the ResNet-18 data pipeline in Phase 2.

In [ ]:
# ── Per-channel mean and std ──────────────────────────────────────────────────
loader = DataLoader(train_dataset, batch_size=512, shuffle=False, num_workers=0)

channel_sum    = torch.zeros(3)
channel_sum_sq = torch.zeros(3)
n_pixels = 0

for imgs, _ in loader:
    # medmnist returns uint8 in shape (B, H, W, C) or (B, C, H, W) — normalise to [0,1]
    imgs = imgs.float()
    if imgs.max() > 1.0:
        imgs = imgs / 255.0
    if imgs.ndim == 4 and imgs.shape[-1] == 3:
        imgs = imgs.permute(0, 3, 1, 2)  # BHWC -> BCHW
    channel_sum    += imgs.sum(dim=[0, 2, 3])
    channel_sum_sq += (imgs ** 2).sum(dim=[0, 2, 3])
    n_pixels       += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]

mean = channel_sum / n_pixels
std  = ((channel_sum_sq / n_pixels) - mean ** 2).sqrt()

print('PathMNIST per-channel stats (normalised to [0,1]):')
print(f'  Mean : R={mean[0]:.4f}  G={mean[1]:.4f}  B={mean[2]:.4f}')
print(f'  Std  : R={std[0]:.4f}   G={std[1]:.4f}   B={std[2]:.4f}')
print()
print('Copy these into transforms.Normalize() in Phase 2 (medforget/datasets/medmnist_wrapper.py).')

## 6. Class-wise Forget Scenario Preview

Simulate the retain/forget split for class 3 and run the invariant checks
(same checks that will be enforced by `tests/test_scenarios.py` in Phase 2).

In [ ]:
# ── Scenario preview: class-wise split for class 3 ───────────────────────────
forget_indices = np.where(train_labels == FORGET_CLASS)[0]
retain_indices = np.where(train_labels != FORGET_CLASS)[0]

# Invariant checks (same as tests/test_scenarios.py)
assert len(np.intersect1d(retain_indices, forget_indices)) == 0, 'OVERLAP BUG!'
assert len(retain_indices) + len(forget_indices) == len(train_dataset), 'NOT EXHAUSTIVE!'

forget_pct = len(forget_indices) / len(train_dataset) * 100
retain_pct = len(retain_indices) / len(train_dataset) * 100

print(f'Forget class  : {FORGET_CLASS} — "{classes[FORGET_CLASS]}"')
print(f'Forget set    : {len(forget_indices):,} samples ({forget_pct:.1f}% of training set)')
print(f'Retain set    : {len(retain_indices):,} samples ({retain_pct:.1f}% of training set)')
print(f'Total         : {len(train_dataset):,} samples')
print()
print('Invariants: disjoint + exhaustive  [PASSED]')

## Summary

| Check | Status |
|---|---|
| PathMNIST loads via `medmnist` | ✓ |
| 9 classes, 28×28 RGB | ✓ |
| Class balance reasonable | ✓ |
| Class 3 (forget target) is well-represented | ✓ |
| Forget/retain split is disjoint + exhaustive | ✓ |
| Image mean/std computed for Phase 2 normalisation | ✓ |

**Phase 0 definition of done:** repo skeleton exists on GitHub, PathMNIST loads and displays correctly. ✓

**Next:** Phase 1 — implement the four abstract base interfaces, `utils/seed.py`, and the example experiment config.
